# Data Quality Test Plan

This notebook defines a practical plan to validate core data quality dimensions using:
1. Great Expectations (framework-based validation), and
2. Custom SQL queries (lightweight and transparent checks).


## 1) Scope and Tables

**Database**: `olist_ecommerce_star.db` (DuckDB file)

**Default schema/table setup in this notebook**:
- `analytics.dim_customer`
- `analytics.dim_product`
- `analytics.dim_seller`
- `analytics.fact_orders`

**Quality categories to test**:
- Null values
- Duplicates
- Referential integrity
- Business logic rules


In [3]:
from pathlib import Path
import duckdb
import pandas as pd

DB_NAME = "olist_ecommerce_star.db"
candidates = [
    Path.cwd() / DB_NAME,
    Path.cwd() / "m2-project" / DB_NAME,
    Path.cwd().parent / DB_NAME,
]

db_path = next((p for p in candidates if p.exists()), None)
if db_path is None:
    raise FileNotFoundError(f"Could not find {DB_NAME}. Checked: {candidates}")

conn = duckdb.connect(str(db_path), read_only=True)
print(f"Connected to DuckDB: {db_path}")

SCHEMA = "analytics"
DIM_CUSTOMER = f"{SCHEMA}.dim_customer"
DIM_PRODUCT = f"{SCHEMA}.dim_product"
DIM_SELLER = f"{SCHEMA}.dim_seller"
FACT_ORDERS = f"{SCHEMA}.fact_orders"

conn


Connected to DuckDB: /home/franklin/dsai/dsai-5m-projects/m2-project/olist_ecommerce_star.db


## 2) Option A: Great Expectations Plan

Use GE expectations to define reusable data contracts per table.

### Suggested expectations
- **Null checks**: `expect_column_values_to_not_be_null`
- **Duplicate checks**: `expect_column_values_to_be_unique` (or compound uniqueness via SQL helper view)
- **Referential integrity**: Use lookup checks with joins or custom expectations
- **Business logic**: Range, set membership, date-order checks


In [5]:
# Optional setup if dependencies are not installed:
# %pip install great-expectations duckdb

import great_expectations as gx
from great_expectations import expectations as gxe

context = gx.get_context()

# Great Expectations 1.8+ pattern: create a pandas datasource, read dataframe as a batch,
# then validate it against an expectation suite.
customers_df = conn.execute(f"SELECT * FROM {DIM_CUSTOMER}").fetchdf()
ds = context.data_sources.add_or_update_pandas(name="pandas_source")
batch = ds.read_dataframe(customers_df)

suite = gx.ExpectationSuite(
    name="dim_customer_suite",
    expectations=[
        gxe.ExpectColumnValuesToNotBeNull(column="customer_pk"),
        gxe.ExpectColumnValuesToBeUnique(column="customer_pk"),
        gxe.ExpectColumnValuesToBeInSet(
            column="customer_state",
            value_set=["AC","AL","AM","AP","BA","CE","DF","ES","GO","MA","MG","MS","MT","PA","PB","PE","PI","PR","RJ","RN","RO","RR","RS","SC","SE","SP","TO"],
        ),
    ],
)

validation_result = batch.validate(suite)
validation_result.success


Calculating Metrics: 100%|██████████| 23/23 [00:00<00:00, 216.93it/s]


True

## 3) Option B: Custom SQL Data Quality Checks

These checks return failing records (or failure counts).
A passing test should return `0` failing rows unless otherwise noted.


In [6]:
def run_check(name: str, sql: str):
    df = conn.execute(sql).fetchdf()
    failures = int(df.iloc[0, 0])
    status = "PASS" if failures == 0 else "FAIL"
    print(f"[{status}] {name}: {failures} failing rows")
    return {"check_name": name, "failures": failures, "status": status}


In [7]:
# 3.1 Null Value Checks
checks = []

checks.append(run_check(
    "dim_customer.customer_pk not null",
    f"""
    SELECT COUNT(*) AS failures
    FROM {DIM_CUSTOMER}
    WHERE customer_pk IS NULL
    """
))

checks.append(run_check(
    "fact_orders.order_id not null",
    f"""
    SELECT COUNT(*) AS failures
    FROM {FACT_ORDERS}
    WHERE order_id IS NULL
    """
))

checks.append(run_check(
    "fact_orders.payment_value not null",
    f"""
    SELECT COUNT(*) AS failures
    FROM {FACT_ORDERS}
    WHERE payment_value IS NULL
    """
))

[PASS] dim_customer.customer_pk not null: 0 failing rows
[PASS] fact_orders.order_id not null: 0 failing rows
[FAIL] fact_orders.payment_value not null: 3 failing rows


In [8]:
# 3.2 Duplicate Checks
checks.append(run_check(
    "dim_customer.customer_pk unique",
    f"""
    SELECT COUNT(*) AS failures
    FROM (
      SELECT customer_pk
      FROM {DIM_CUSTOMER}
      GROUP BY customer_pk
      HAVING COUNT(*) > 1
    ) d
    """
))

checks.append(run_check(
    "fact_orders (order_id, order_item_id) unique",
    f"""
    SELECT COUNT(*) AS failures
    FROM (
      SELECT order_id, order_item_id
      FROM {FACT_ORDERS}
      GROUP BY order_id, order_item_id
      HAVING COUNT(*) > 1
    ) d
    """
))

[PASS] dim_customer.customer_pk unique: 0 failing rows
[FAIL] fact_orders (order_id, order_item_id) unique: 3309 failing rows


In [9]:
# 3.3 Referential Integrity Checks
checks.append(run_check(
    "fact_orders.customer_fk exists in dim_customer.customer_pk",
    f"""
    SELECT COUNT(*) AS failures
    FROM {FACT_ORDERS} f
    LEFT JOIN {DIM_CUSTOMER} d ON f.customer_fk = d.customer_pk
    WHERE f.customer_fk IS NOT NULL
      AND d.customer_pk IS NULL
    """
))

checks.append(run_check(
    "fact_orders.product_fk exists in dim_product.product_pk",
    f"""
    SELECT COUNT(*) AS failures
    FROM {FACT_ORDERS} f
    LEFT JOIN {DIM_PRODUCT} p ON f.product_fk = p.product_pk
    WHERE f.product_fk IS NOT NULL
      AND p.product_pk IS NULL
    """
))

checks.append(run_check(
    "fact_orders.seller_fk exists in dim_seller.seller_pk",
    f"""
    SELECT COUNT(*) AS failures
    FROM {FACT_ORDERS} f
    LEFT JOIN {DIM_SELLER} s ON f.seller_fk = s.seller_pk
    WHERE f.seller_fk IS NOT NULL
      AND s.seller_pk IS NULL
    """
))

[PASS] fact_orders.customer_fk exists in dim_customer.customer_pk: 0 failing rows
[PASS] fact_orders.product_fk exists in dim_product.product_pk: 0 failing rows
[PASS] fact_orders.seller_fk exists in dim_seller.seller_pk: 0 failing rows


In [10]:
# 3.4 Business Logic Checks
checks.append(run_check(
    "payment_value >= 0",
    f"""
    SELECT COUNT(*) AS failures
    FROM {FACT_ORDERS}
    WHERE payment_value < 0
    """
))

checks.append(run_check(
    "freight_value >= 0",
    f"""
    SELECT COUNT(*) AS failures
    FROM {FACT_ORDERS}
    WHERE freight_value < 0
    """
))

checks.append(run_check(
    "payment_installments > 0 when payment_value > 0",
    f"""
    SELECT COUNT(*) AS failures
    FROM {FACT_ORDERS}
    WHERE payment_value > 0
      AND (payment_installments IS NULL OR payment_installments <= 0)
    """
))

checks.append(run_check(
    "delivered orders should have customer delivery date key",
    f"""
    SELECT COUNT(*) AS failures
    FROM {FACT_ORDERS}
    WHERE order_status = 'delivered'
      AND order_delivered_customer_date_fk IS NULL
    """
))

[PASS] payment_value >= 0: 0 failing rows
[PASS] freight_value >= 0: 0 failing rows
[FAIL] payment_installments > 0 when payment_value > 0: 3 failing rows
[PASS] delivered orders should have customer delivery date key: 0 failing rows


In [11]:
results_df = pd.DataFrame(checks)
results_df


,check_name,failures,status
0,dim_customer.customer_pk not null,0,PASS
1,fact_orders.order_id not null,0,PASS
2,fact_orders.payment_value not null,3,FAIL
3,dim_customer.customer_pk unique,0,PASS
4,"fact_orders (order_id, order_item_id) unique",3309,FAIL
5,fact_orders.customer_fk exists in dim_customer...,0,PASS
6,fact_orders.product_fk exists in dim_product.p...,0,PASS
7,fact_orders.seller_fk exists in dim_seller.sel...,0,PASS
8,payment_value >= 0,0,PASS
9,freight_value >= 0,0,PASS


## 4) Execution Plan

1. Run all SQL checks and capture baseline failure counts.
2. Convert stable checks into Great Expectations suites for repeatability.
3. Define thresholds for warnings vs hard failures (if needed).
4. Publish results to a data quality report per run date.
5. Integrate notebook logic into scheduled ETL/ELT validation pipeline.


## 5) Notes

- If table names differ in your environment, update SQL accordingly.
- For production pipelines, move SQL rules into version-controlled `.sql` files and parameterize runtime.
- Keep both GE and SQL checks: GE for governance and SQL for transparent debugging.
